# Q1: Multi-Task CNN on Fashion-MNIST

All experiment logs are available on [Weights & Biases](https://wandb.ai/sudhujoshi-smai/smai-a4-q1/workspace?nw=nwusersudhujoshi).

In [1]:
# Imports necessary libraries along with folder cleanup and wandb login.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import time
import os
import shutil
import nbformat
import torch

# Cell 2: Additional Imports and Setup
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
import wandb
import copy

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", 
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# 1. Checks for CUDA & PyTorch
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Using GPU: {gpu_name}")
else:
    DEVICE = torch.device("cpu")
    print("No GPU found. Using CPU.")
print(f"Using device object: {DEVICE}")


artifacts_dir = 'runs-q1'
if os.path.exists(artifacts_dir):
    print(f"Removing old artifacts directory: {artifacts_dir}")
    shutil.rmtree(artifacts_dir)
print(f"Creating clean artifacts directory: {artifacts_dir}")
os.makedirs(artifacts_dir)

# Define the local wandb directory name
wandb_dir = 'wandb'
PROJECT_NAME = 'smai-a4-q1'
# Check if the directory exists and remove it
if os.path.exists(wandb_dir):
    print(f"Removing old wandb directory: {wandb_dir}")
    shutil.rmtree(wandb_dir)
    
import wandb
wandb.login()

# Fix seed for reproducible randomization.
username = "sudhanva.joshi"
seed = int(hashlib.sha256(username.encode()).hexdigest(), 16) % (2**32)
print("Seed generated:", seed)
rng = np.random.default_rng(seed)
np.random.seed(seed) 

# Set plotting theme and helper functons.
sns.set_theme(context="paper", style="darkgrid",
              palette="magma", font="Nimbus Roman")

def plotinfo(title, x, y, userid=username):
    plt.text(0.95, -0.1, userid, ha="right", va="top",
             transform=plt.gca().transAxes, fontsize=10,
             color="gray", alpha=0.7)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel(x, fontsize=12)
    plt.ylabel(y, fontsize=12)
    plt.tight_layout()
    plt.show()

def coordplot(ax, title, x_label, y_label):
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)

PyTorch version: 2.5.1+cu121
Using GPU: NVIDIA GeForce GTX 1650
Using device object: cuda
Removing old artifacts directory: runs-q1
Creating clean artifacts directory: runs-q1
Removing old wandb directory: wandb


wandb: Currently logged in as: sudhujoshi (sudhujoshi-smai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Seed generated: 1027714457


## Data Loading & Preprocessing

In [2]:
class FashionMNISTDataset(Dataset):
    """
    Custom Dataset for Fashion-MNIST Multi-Task Learning.
    Returns (image, class_label, ink_target)
    """
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # Get the original PIL image and label
        image, label = self.dataset[idx] 
        
        # Calculate ink_target: normalized average pixel value 
        # We use the original PIL image before normalization for this.
        # Convert PIL to numpy -> scale to [0, 1] -> get mean
        ink_target = torch.tensor(np.array(image).mean() / 255.0, dtype=torch.float32)

        # Apply transformations (normalization, augmentation)
        if self.transform:
            image = self.transform(image)
        
        return image, label, ink_target


def get_dataloaders(batch_size, train_val_split_ratio=0.9):
    """
    Loads Fashion-MNIST, calculates mean/std, applies transforms,
    and returns train, val, and test dataloaders.
    """
    
    # 1. Load full training data to calculate mean/std
    # We load it first with a simple ToTensor transform
    temp_transform = transforms.ToTensor()
    temp_train_data = FashionMNIST(root='./data', train=True, download=True, transform=temp_transform)
    
    print("Calculating dataset mean and std.")
    data_stack = torch.stack([img for img, _ in temp_train_data], dim=0)
    mean = data_stack.mean()
    std = data_stack.std()
    print(f"Calculated Mean: {mean}, Std: {std}")

    # 2. Define Transforms
    train_transform = transforms.Compose([
        transforms.RandomRotation(10),
        transforms.RandomCrop(28, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((mean,), (std,))
    ])
    
    # No augmentations for val and test sets
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((mean,), (std,))
    ])

    # 3. Create Datasets
    # Load full train data (this time as PIL images for our custom class)
    full_train_dataset_pil = FashionMNIST(root='./data', train=True, download=True)
    
    # Split train/val sets.
    n_train = int(len(full_train_dataset_pil) * train_val_split_ratio)
    n_val = len(full_train_dataset_pil) - n_train
    
    train_indices, val_indices = random_split(range(len(full_train_dataset_pil)), [n_train, n_val])
    
    # Create custom datasets with the correct transforms
    train_data = FashionMNISTDataset(
        Subset(full_train_dataset_pil, train_indices), 
        transform=train_transform
    )
    val_data = FashionMNISTDataset(
        Subset(full_train_dataset_pil, val_indices), 
        transform=test_transform
    )
    
    # Create test dataset
    test_data_pil = FashionMNIST(root='./data', train=False, download=True)
    test_data = FashionMNISTDataset(test_data_pil, transform=test_transform)
    
    print(f"Train samples: {len(train_data)}")
    print(f"Validation samples: {len(val_data)}")
    print(f"Test samples: {len(test_data)}")

    # 4. Create DataLoaders
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader, test_loader, mean, std

## Model Implementation: Multi-Task CNN

In [3]:
class MultiTaskCNN(nn.Module):
    """
    Multi-task CNN for Fashion-MNIST classification and ink-value regression.
    """
    def __init__(self, dropout_rate=0.25, num_blocks=3):
        super(MultiTaskCNN, self).__init__()
        
        if num_blocks not in [2, 3]:
            raise ValueError("num_blocks must be 2 or 3")
        self.num_blocks = num_blocks
        
        # Block 1
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(dropout_rate)
        ) # Output: [B, 32, 14, 14]
        
        # Block 2 (Always included)
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(dropout_rate)
        ) # Output: [B, 64, 7, 7]
        
        # Block 3
        if self.num_blocks == 3:
            self.conv_block3 = nn.Sequential(
                nn.Conv2d(64, 128, kernel_size=3, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2), # Output: [B, 128, 3, 3]
                nn.Dropout(dropout_rate)
            )
            # Flattened size for 3 blocks
            self.flattened_size = 128 * 3 * 3
        else:
            # Flattened size for 2 blocks
            self.flattened_size = 64 * 7 * 7

        # 1. Classification Head
        self.classification_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 10) # 10 classes
        )
        
        # 2. Regression Head
        self.regression_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1), 
            nn.Sigmoid()   # Bound output to [0, 1]
        )
        
        # Hooks for Feature Map Visualization
        self.feature_maps = {}
        self.conv_block1[2].register_forward_hook(self.get_activation('block1_relu'))
        self.conv_block2[2].register_forward_hook(self.get_activation('block2_relu'))
        if self.num_blocks == 3:
            self.conv_block3[2].register_forward_hook(self.get_activation('block3_relu')) 
        
    def get_activation(self, name):
        """Helper to create a hook function for saving activations"""
        def hook(model, input, output):
            self.feature_maps[name] = output.detach().cpu()
        return hook

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        if self.num_blocks == 3:
            x = self.conv_block3(x)
        
        # Pass features to both heads
        class_logits = self.classification_head(x)
        regression_output = self.regression_head(x)
        
        return class_logits, regression_output

## Hyperparameter Tuning & WandB Logging

Since the current task entails classification, I observed that $\lambda_{1}$ is more relevant as a loss function compared to $\lambda_{2}$. Since the task is mainly classification oriented, this shows that the Cross-Entropy term is a stronger choice for optimization, and the sweeps below have the same preference as well to optimize for accuracy.

[This](https://wandb.ai/sudhujoshi-smai/smai-a4-q1/runs/5oofzhdq?nw=nwusersudhujoshi) run had the best accuracy, while [this](https://wandb.ai/sudhujoshi-smai/smai-a4-q1/runs/t9txmntj?nw=nwusersudhujoshi) had the lowest RMSE.

In [4]:
# Cell 5: Training and Evaluation Logic

def train_one_epoch(model, loader, optimizer, criterion_ce, criterion_mse, lambda1, lambda2, device, epoch):
    model.train()
    total_loss_epoch = 0
    total_ce_loss_epoch = 0
    total_mse_loss_epoch = 0
    
    for i, (images, labels, ink_targets) in enumerate(tqdm(loader, desc=f"Epoch {epoch+1} Train")):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        ink_targets = ink_targets.to(DEVICE)
        
        # Forward pass
        logits, regression = model(images)
        
        # Calculate individual losses
        loss_ce = criterion_ce(logits, labels)
        loss_mse = criterion_mse(regression.squeeze(), ink_targets)
        
        # Calculate joint loss
        total_loss = (lambda1 * loss_ce) + (lambda2 * loss_mse)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        # Log batch metrics to wandb
        batch_metrics = {
            "train/batch/total_loss": total_loss.item(),
            "train/batch/ce_loss": loss_ce.item(),
            "train/batch/mse_loss": loss_mse.item(),
            "epoch": epoch + 1
        }
        wandb.log(batch_metrics)
        
        total_loss_epoch += total_loss.item()
        total_ce_loss_epoch += loss_ce.item()
        total_mse_loss_epoch += loss_mse.item()

    # Return average epoch losses
    n_batches = len(loader)
    return (total_loss_epoch / n_batches, 
            total_ce_loss_epoch / n_batches, 
            total_mse_loss_epoch / n_batches)


def validate_one_epoch(model, loader, criterion_ce, criterion_mse, lambda1, lambda2, device, epoch):
    model.eval()
    all_labels = []
    all_preds = []
    all_ink_targets = []
    all_ink_preds = []
    
    total_val_loss = 0
    total_val_ce = 0
    total_val_mse = 0
    
    with torch.no_grad():
        for images, labels, ink_targets in tqdm(loader, desc=f"Epoch {epoch+1} Val"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            ink_targets = ink_targets.to(DEVICE)
            
            # Forward pass
            logits, regression = model(images)
            
            # Calculate losses
            loss_ce = criterion_ce(logits, labels)
            loss_mse = criterion_mse(regression.squeeze(), ink_targets)
            total_loss = (lambda1 * loss_ce) + (lambda2 * loss_mse)
            
            total_val_loss += total_loss.item()
            total_val_ce += loss_ce.item()
            total_val_mse += loss_mse.item()
            
            # Store predictions and targets for metric calculation
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_ink_targets.extend(ink_targets.cpu().numpy())
            all_ink_preds.extend(regression.squeeze().cpu().numpy())

    # Calculate validation metrics
    n_batches = len(loader)
    avg_val_loss = total_val_loss / n_batches
    avg_val_ce = total_val_ce / n_batches
    avg_val_mse = total_val_mse / n_batches
    
    accuracy = np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)
    mae = mean_absolute_error(all_ink_targets, all_ink_preds)
    rmse = np.sqrt(mean_squared_error(all_ink_targets, all_ink_preds))
    
    # Log epoch metrics to wandb
    epoch_metrics = {
        "val/total_loss": avg_val_loss,
        "val/ce_loss": avg_val_ce,
        "val/mse_loss": avg_val_mse,
        "val/accuracy": accuracy,
        "val/mae": mae,
        "val/rmse": rmse,
        "epoch": epoch + 1
    }
    wandb.log(epoch_metrics)
    
    return avg_val_loss, accuracy, rmse

### Feature Map Visualization

Each block helped in dimensionality reduction and capturing various features. 

Block 1 mainly captured major edges of the object.
Block 2 captured complexities and intricacies in the image.
Block 3 had a bottleneck due to repeating pooling/convolution, and gave very coarse pixelated instances. This would be relevant if our images had a higher resolution, but it helps us to capture general spaces occupied by the object in the image nonetheless.

In [5]:
def unnormalize(tensor, mean, std):
    """Un-normalizes a tensor."""
    # Ensure mean/std are tensors
    mean = torch.tensor(mean).view(1, -1, 1, 1)
    std = torch.tensor(std).view(1, -1, 1, 1)
    return tensor.cpu() * std + mean


def plot_feature_maps(model, loader, device, wandb_log=True, mean=0.286, std=0.353):
    """
    Visualizes feature maps for one sample from each of the 10 classes.
    Adds layer names as row subheadings.
    Logs all 10 images to wandb in a single call.
    """
    model.eval()
    
    # 1. Collect one sample for each class
    samples_by_class = {}
    print("Collecting one sample from each class for visualization...")
    
    for images, labels, ink_targets in loader:
        for i in range(len(labels)):
            label = labels[i].item()
            if label not in samples_by_class:
                samples_by_class[label] = (images[i], labels[i], ink_targets[i])
            if len(samples_by_class) == 10:
                break
        if len(samples_by_class) == 10:
            break
            
    print(f"Collected {len(samples_by_class)} unique class samples.")

    log_dictionary = {}
    
    # 2. Plot feature maps for each class
    for label in sorted(samples_by_class.keys()):
        image, label_tensor, ink_target = samples_by_class[label]
        class_name = CLASS_NAMES[label]
        ink_val = ink_target.item()
        
        # 3. Run forward pass on this single image
        with torch.no_grad():
            _ = model(image.unsqueeze(0).to(device))
        
        # Get the feature maps saved by the hooks
        feature_maps = model.feature_maps 
        
        if not feature_maps:
            print(f"Warning: No feature maps found for {class_name}.")
            continue
            
        num_layers = len(feature_maps)
        maps_to_plot = 8  # Plot 8 maps per layer
        
        # Create a subplot grid: N_layers rows, (1 + 8) columns
        # The first column is for the original image and layer names
        fig, axs = plt.subplots(
            num_layers, 
            maps_to_plot + 1, 
            figsize=(16, 2 + num_layers * 2), # Taller plot
            gridspec_kw={'width_ratios': [2] + [1]*maps_to_plot}
        )
        
        fig.suptitle(f"Feature Maps - {class_name} (Ink: {ink_val:.3f})", fontsize=16)

        # FIX: The following loop must be inside the main sample loop
        for layer_idx, (layer_name, f_map) in enumerate(feature_maps.items()):
            
            # Handle case where num_layers (i.e., num_blocks) is 1
            if num_layers == 1:
                # If only one layer, axs is a 1D array of subplots
                ax_img = axs[0]
                ax_maps_start_idx = 1
            else:
                # If multiple layers, axs is a 2D array [row, col]
                ax_img = axs[layer_idx, 0]
                ax_maps_start_idx = 1
            
            # 1. ALWAYS ADD THE LAYER NAME SUBHEADING (Y-LABEL)
            ax_img.set_ylabel(layer_name, fontsize=12, fontweight='bold', labelpad=20)
            
            # 2. Plot the original image in the first column (only for the first row)
            if layer_idx == 0:
                # Use the unnormalize helper function
                denorm_img = unnormalize(image.unsqueeze(0), mean, std)
                ax_img.imshow(denorm_img[0].permute(1, 2, 0).squeeze(), cmap='gray')
                ax_img.set_title("Original Image", fontsize=10)
            else:
                # 3. For subsequent rows, clear the plot area while keeping the Y-label
                ax_img.set_facecolor('white') 
                ax_img.spines['top'].set_visible(False)
                ax_img.spines['right'].set_visible(False)
                ax_img.spines['bottom'].set_visible(False)
                ax_img.spines['left'].set_visible(False)
            
            # 4. Clear ticks/labels for the first column (applies to all rows)
            ax_img.set_xticks([])
            ax_img.set_yticks([])

            # Plot the 8 feature maps
            f_map_batch_removed = f_map.squeeze(0) # Remove batch dim [C, H, W]
            for col_idx in range(maps_to_plot):
                # Determine the correct subplot object for the feature map
                if num_layers == 1:
                    ax = axs[ax_maps_start_idx + col_idx]
                else:
                    ax = axs[layer_idx, ax_maps_start_idx + col_idx]

                if col_idx < f_map_batch_removed.shape[0]: # Check if map exists
                    ax.imshow(f_map_batch_removed[col_idx].numpy(), cmap='magma')
                ax.set_xticks([])
                ax.set_yticks([])
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust for suptitle
        
        # Add username
        plt.text(0.95, 0.01, username, ha='right', va='bottom',
                 transform=fig.transFigure, fontsize=10, color='gray', alpha=0.7)

        if wandb_log:
            wandb_log_name = f"val/feature_maps/Class_{label}_{class_name}"
            
            # Assign the image object directly to a key in the summary.
            wandb.run.summary[wandb_log_name] = wandb.Image(fig)
        
        plt.close(fig) # Close fig to save memory
    
    print("Feature map generation complete.")

In [6]:
def run_experiment(config):
    """
    Main function to run a single experiment with a given config.
    """
    rounded_lr = f"{config['lr']:.3g}"
    
    run_name = (f"l1_{config['lambda1']}_l2_{config['lambda2']}_lr_{rounded_lr}_"
                f"bs_{config['batch_size']}_do_{config['dropout']}_"
                f"opt_{config['optimizer']}_blks_{config['num_blocks']}")

    wandb.init(
        project=PROJECT_NAME,
        config=config,
        name=run_name,
        reinit=True
    )
    
    print(f"\\n Starting Run: {run_name}")
    
    # Get config details
    cfg = wandb.config
    
    # 2. Load Data
    train_loader, val_loader, test_loader, mean, std = get_dataloaders(batch_size=cfg.batch_size)
    
    # 3. Initialize Model, Optimizer, Loss
    model = MultiTaskCNN(
        dropout_rate=cfg.dropout, 
        num_blocks=cfg.num_blocks
    ).to(DEVICE)
    
    wandb.watch(model, log='all', log_freq=100)
    
    if cfg.optimizer == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=cfg.lr, momentum=0.9)
    elif cfg.optimizer == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=cfg.lr)
    else:
        optimizer = optim.AdamW(model.parameters(), lr=cfg.lr)
        
    criterion_ce = nn.CrossEntropyLoss()
    criterion_mse = nn.MSELoss()
    
    # 4. Training Loop
    best_val_loss = float('inf')
    best_val_acc = 0.0
    best_val_rmse = float('inf')
    
    best_model_loss = None
    best_model_acc = None
    best_model_rmse = None
    
    for epoch in range(cfg.epochs):
        # Train
        train_loss, train_ce, train_mse = train_one_epoch(
            model, train_loader, optimizer, criterion_ce, criterion_mse, 
            cfg.lambda1, cfg.lambda2, DEVICE, epoch
        )
        
        # Log train epoch metrics
        wandb.log({
            "train/epoch/total_loss": train_loss,
            "train/epoch/ce_loss": train_ce,
            "train/epoch/mse_loss": train_mse,
            "epoch": epoch + 1
        })
        
        # Validate
        val_loss, val_acc, val_rmse = validate_one_epoch(
            model, val_loader, criterion_ce, criterion_mse, 
            cfg.lambda1, cfg.lambda2, DEVICE, epoch
        )
        
        print(f"Epoch {epoch+1}: Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val RMSE: {val_rmse:.4f}")
        
        # 5. Model Selection
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_loss = copy.deepcopy(model.state_dict())
            wandb.run.summary["best_val_loss"] = best_val_loss
            
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_acc = copy.deepcopy(model.state_dict())
            wandb.run.summary["best_val_accuracy"] = best_val_acc

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_model_rmse = copy.deepcopy(model.state_dict())
            wandb.run.summary["best_val_rmse"] = best_val_rmse

    print(f"Run {run_name} Finished.")
    print(f"Best Val Accuracy: {best_val_acc:.4f}")
    print(f"Best Val RMSE: {best_val_rmse:.4f}")

    # 6. Final Evaluation on Test Set
    print("Evaluating on Test Set.")
    models_to_test = {
        "best_acc": best_model_acc,
        "best_rmse": best_model_rmse,
        "best_loss": best_model_loss
    }
    
    for model_name, model_state in models_to_test.items():
        if model_state is None: continue
        
        model.load_state_dict(model_state)
        test_loss, test_acc, test_rmse = validate_one_epoch(
            model, test_loader, criterion_ce, criterion_mse,
            cfg.lambda1, cfg.lambda2, DEVICE, epoch=-1
        )
        
        wandb.run.summary[f"test_accuracy_on_{model_name}_model"] = test_acc
        wandb.run.summary[f"test_rmse_on_{model_name}_model"] = test_rmse
        wandb.run.summary[f"test_loss_on_{model_name}_model"] = test_loss
        print(f"Test Metrics (using {model_name} model): Acc: {test_acc:.4f}, RMSE: {test_rmse:.4f}")

    # 7. Log Final Visualizations
    print("Logging final visualizations.")
    if best_model_acc is not None:
        model.load_state_dict(best_model_acc)
        plot_feature_maps(model, test_loader, DEVICE, wandb_log=True, mean=mean, std=std)
    
    # 8. Finish Run
    wandb.finish()

In [7]:
# 1. Define the Sweep Configuration
sweep_config = {
    'method': 'bayes',  # Bayesian optimization
    'metric': {
        'name': 'val/total_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'epochs': {
            'value': 20  # Fixed value
        },
        'batch_size': {
            'values': [64, 128]
        },
        'lr': {
            'distribution': 'log_uniform_values',
            'min': 1e-4,
            'max': 3e-3
        },
        'optimizer': {
            'values': ['AdamW', 'SGD', 'RMSprop']
        },
        'dropout': {
            'values': [0.25, 0.5]
        },
        'num_blocks': {
            'values': [2, 3]  # Sweeping over block sizes
        },
        'lambda1': {
            'values': [1.0, 10.0]  # Emphasize classification
        },
        'lambda2': {
            'values': [1.0, 10.0]  # Emphasize regression
        }
    }
}

def sweep_agent_run():
    wandb.init()  # Initialize the run
    config = wandb.config  # Get the config for this run
    
    # Create the full config dictionary expected by run_experiment
    run_config = {
        "epochs": config.epochs,
        "batch_size": config.batch_size,
        "optimizer": config.optimizer,
        "dropout": config.dropout,
        "lambda1": config.lambda1,
        "lambda2": config.lambda2,
        "lr": config.lr,
        "num_blocks": config.num_blocks
    }
    
    run_experiment(run_config)


# 3. Initialize the Sweep
sweep_id = wandb.sweep(sweep_config, project=PROJECT_NAME)

# 4. Run Agent.
print(f"Running sweep agent with ID: {sweep_id}")
wandb.agent(sweep_id, function=sweep_agent_run, count=15)

print("\\n All sweep experiments completed.")

Create sweep with ID: d9pz7xyz
Sweep URL: https://wandb.ai/sudhujoshi-smai/smai-a4-q1/sweeps/d9pz7xyz
Running sweep agent with ID: d9pz7xyz


wandb: Agent Starting Run: 5oofzhdq with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.25
wandb: 	epochs: 20
wandb: 	lambda1: 10
wandb: 	lambda2: 1
wandb: 	lr: 0.0026546150062309102
wandb: 	num_blocks: 3
wandb: 	optimizer: SGD


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


\n Starting Run: l1_10_l2_1_lr_0.00265_bs_128_do_0.25_opt_SGD_blks_3
Calculating dataset mean and std.
Calculated Mean: 0.2860405743122101, Std: 0.3530242443084717
Train samples: 54000
Validation samples: 6000
Test samples: 10000


Epoch 1 Val: 100%|██████████| 47/47 [00:01<00:00, 39.42it/s]


Epoch 1: Val Loss: 5.9072, Val Acc: 0.7725, Val RMSE: 0.0658


... (769 outputs left out)

/tmp/ipykernel_199983/605918456.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mean = torch.tensor(mean).view(1, -1, 1, 1)
/tmp/ipykernel_199983/605918456.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  std = torch.tensor(std).view(1, -1, 1, 1)


Feature map generation complete.


epoch,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇█████
train/batch/ce_loss,▆█▃▄▅▃▄▃▂▂▃▃▃▃▄▂▂▄▂▁▃▃▃▂▂▂▂▁▂▁▄▂▂▁▂▄▃▃▄▃
train/batch/mse_loss,▅▄▂▄▂▂▁▂▁▃▁▃▂▃▂▃▁▂▄▃▄▃▅▃▃▁▃▄▄▃▂▂▃▄█▃▃▂▄▃
train/batch/total_loss,█▃▅▅▃▂▃▃▃▃▃▃▃▃▁▃▂▂▂▂▂▂▃▂▂▃▂▂▃▃▁▁▂▁▂▁▂▁▁▂
train/epoch/ce_loss,█▅▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/epoch/mse_loss,█▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch/total_loss,█▅▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val/accuracy,▁▂▄▄▅▆▆▆▇▇▇▇▇▇██▇▇███▆█
val/ce_loss,█▇▅▄▄▃▂▃▂▂▂▂▂▂▁▁▂▂▁▁▁▃▁
val/mae,▂▆▅▂▄▆▃▄▁▂▂▄█▅▃▄▆▆▆▅▆▁▆
+3,...


\n All sweep experiments completed.
